# Coordinate System and Plotting

**Part I · Visualization** — Tutorial 10

Turn the viewer into a 2D/3D plotting tool with `CoordinateSystem` — a helper
that builds a complete plotting coordinate system (grid, axes with value
labels, an optional background plane, and plotted `PointPath`s) inside a single
`VizGroup`. You will learn to:

- Configure the data range (`xlim` / `ylim`) and linear/log scales.
- Map between data and world coordinates (`plot`, `to_world`, `to_data`).
- Add annotations (`vline`, `hline`, `line`, `point`).
- Register live trails (`add_plot` / `update_plots`) and update ranges in place.


## Setup


In [ ]:
import math

from pytanga.geometry import Point
from pytanga.viz import (
    CoordinateSystem, LinearScale, LogScale, PointPath, PointPathStyle, Scale, Visualizer,
)


## 1. Quick start — a log-log plot

Pass `xscale="log"` / `yscale="log"` for logarithmic axes (the world stays
linear; tick positions and labels are computed in Python).


In [ ]:
viz = Visualizer(
    title="CoordinateSystem — log-log",
    space_dim=2,
    add_default_axes=False,
    add_default_grid=False,
)

cs = CoordinateSystem(viz, xlim=(0.1, 1000), ylim=(1, 1e6), xscale="log", yscale="log")

xs = [0.1 * (10 ** (0.1 * i)) for i in range(40)]
cs.plot(xs, [x * x for x in xs], color="#ffcc00", style=PointPathStyle(line_thickness=3))

viz.display_snapshot()


## 2. Data range and scales

`xlim` / `ylim` set the data range; `xscale` / `yscale` accept `"linear"`,
`"log"`, or a `Scale` instance. The scale classes are available directly
(`LogScale(base)`). `base` sets the log base when a scale is given as `"log"`.


In [ ]:
print("log2 of 8 =", LogScale(2).to_world(8))

viz = Visualizer(title="CoordinateSystem — base-2 log", space_dim=2, add_default_axes=False, add_default_grid=False)
cs = CoordinateSystem(viz, xlim=(1, 64), ylim=(1, 64), xscale="log", yscale="log", base=2)

xs = [1.2 * (2 ** (0.1 * i)) for i in range(30)]
cs.plot(xs, [x * x for x in xs], color="#44ff44")
viz.display_snapshot()


## 3. External `size`, `align` / `axis_origin`, and 3D placement

`size` sets the **physical** extent of the plot plane (independent of the data
range). `align` places the plane relative to `position`; `axis_origin` is the
data point where the axes cross. In 3D, `position` / `normal` / `up` place and
orient the plane (all also accept `Point()` / `Direction()`).


In [ ]:
viz = Visualizer(title="CoordinateSystem — 3D plane", add_default_axes=False, add_default_grid=False)

cs = CoordinateSystem(
    viz,
    xlim=(0, 2 * math.pi),
    ylim=(-1.5, 1.5),
    size=(2.0, 1.0),                 # plane is 2×1 world units; data is stretched onto it
    labels=("x", "sin(x)"),
    position=(-2.0, 1.5, -2.0),
    normal=(1.0, 0.0, 1.0),
    up=(0.0, 1.0, 0.0),
    align=(0, 0),
    axis_origin=(0, 0),
)

xs = [0.05 * i for i in range(int(2 * math.pi / 0.05) + 1)]
cs.plot(xs, [math.sin(x) for x in xs], color="#44ff44", style=PointPathStyle(line_thickness=3))
viz.display_snapshot()


## 4. Mapping data ↔ world coordinates

`plot()` maps data series through the scales automatically. Use `to_world()` /
`to_local()` / `transform()` for explicit conversions, and `data_group` +
`to_data(x, y)` to draw directly in data coordinates (Python-side scaling is
only needed for log axes).


In [ ]:
viz = Visualizer(title="CoordinateSystem — mapping", space_dim=2, add_default_axes=False, add_default_grid=False)
cs = CoordinateSystem(viz, xlim=(0, 10), ylim=(0, 10), labels=("x", "y"))

print("to_world(5, 5):", cs.to_world(5, 5))
print("to_data(5, 5):", cs.to_data(5, 5))

# Draw a custom path directly in the data group:
spike = PointPath()
spike.add((2, 2)); spike.add((2, 8))
cs.data_group.new(spike, color="#ffffff", style=PointPathStyle(line_thickness=1))
viz.display_snapshot()


## 5. Annotations — `vline`, `hline`, `line`, `point`

Each helper accepts `(x, y)` tuples or `Point` instances, creates-or-updates by
an optional `name`, and is removable (`remove_vline`, `remove_hline`,
`remove_line`, `remove_point`) and labelable (`label` + `label_style`).


In [ ]:
from pytanga.viz import LabelStyle, PointStyle

viz = Visualizer(title="CoordinateSystem — annotations", space_dim=2, add_default_axes=False, add_default_grid=False)
cs = CoordinateSystem(viz, xlim=(0, 4 * math.pi), ylim=(-1.5, 1.5), labels=("x", "sin(x)"))

xs = [0.05 * i for i in range(int(4 * math.pi / 0.05) + 1)]
cs.plot(xs, [math.sin(x) for x in xs], color="#44ff44")

cs.vline(x=math.pi, name="pi", color="#ff5555", label="π")
cs.hline(y=0.0, name="zero", color="#8888ff", label_style=LabelStyle(along=0.2))
cs.line((1.0, -1.0), (3.0, 1.0), color="#ff88ff")
cs.point((math.pi, math.sin(math.pi)), color="#ffffff")
cs.point(Point(2 * math.pi, 0.0), name="peak", color="#ffaa00", style=PointStyle(size=0.1), label="2π")

viz.display_snapshot()


## 6. Live trails — `add_plot()` + `update_plots()`

Register a `PointPath` (in data coordinates) and re-sync it each frame. With
`auto_x=True`, the x axis auto-fits to the trail's current x range (minimum span
`min_x_span`), useful for a live time axis.


In [ ]:
viz = Visualizer(title="CoordinateSystem — live trail", space_dim=2, add_default_axes=False, add_default_grid=False)
cs = CoordinateSystem(viz, xlim=(0, 12), ylim=(-1.2, 1.2), labels=("t", "value"))

trail = PointPath(max_points=600)
cs.add_plot(trail, color="#ffcc00", style=PointPathStyle(line_thickness=2), auto_x=True)

t = 0.0
for _ in range(120):
    t += 0.1
    trail.add((t, math.sin(t)))
    cs.update_plots()
    viz.flush()

viz.display_snapshot()
print("trail recorded (120 samples)")


## 7. Updating ranges / scales in place

Change ranges and scales by assignment — children are rebuilt **in place**
(same object ids), so the scene updates without re-adding objects.


In [ ]:
viz = Visualizer(title="CoordinateSystem — in-place", space_dim=2, add_default_axes=False, add_default_grid=False)
cs = CoordinateSystem(viz, xlim=(0, 10), ylim=(0, 10), labels=("x", "y"))
cs.plot([1, 2, 3], [1, 4, 9], color="#44ff44")

cs.xlim = (0, 100)          # rescale the x axis (grid + axes + labels update)
cs.ylim = (1, 100)          # positive range required before switching to log
cs.yscale = "log"           # switch the y axis to log
cs.base = 2                 # change the log base of both log axes
cs.size = (4, 2)            # change the external world/plane extent
cs.align = (0, 0)           # move the plane so its bottom-left corner is at position

viz.display_snapshot()


## 8. Styling the parts

Style the parts at construction via `x_style` / `y_style` (`AxisStyle`),
`grid_style` (`GridStyle`), and `plane_style` (`PlaneStyle`). The group itself
is exposed as `cs.group`.


## Visual Examples

Log-log, 3D plane, and live-trail plots exported via `export_snapshot()`.


In [ ]:
# Log-log plot
viz = Visualizer(title="Plot — log-log", space_dim=2, add_default_axes=False, add_default_grid=False)
cs = CoordinateSystem(viz, xlim=(0.1, 1000), ylim=(1, 1e6), xscale="log", yscale="log", labels=("f", "P"))
xs = [0.1 * (10 ** (0.1 * i)) for i in range(40)]
cs.plot(xs, [x * x for x in xs], color="#ffcc00", style=PointPathStyle(line_thickness=3))
cs.vline(x=100, name="f", color="#44aaff")
viz.display_snapshot()

# 3D tilted-plane plot
viz3 = Visualizer(title="Plot — 3D plane", add_default_axes=False, add_default_grid=False)
cs3 = CoordinateSystem(viz3, xlim=(0, 2 * math.pi), ylim=(-1.5, 1.5), size=(2.0, 1.0),
                       labels=("x", "sin(x)"), position=(-2, 1.5, -2), normal=(1, 0, 1), up=(0, 1, 0))
xs3 = [0.05 * i for i in range(int(2 * math.pi / 0.05) + 1)]
cs3.plot(xs3, [math.sin(x) for x in xs3], color="#44ff44", style=PointPathStyle(line_thickness=3))
viz3.display_snapshot()


## Summary

| Task | API |
|---|---|
| Build a plot | `CoordinateSystem(viz, xlim=..., ylim=..., xscale=..., yscale=...)` |
| Log scale | `xscale="log"` / `LogScale(base)` |
| Plot a series | `cs.plot(xs, ys, color=..., style=...)` |
| Data → world | `cs.to_world(x, y)` / `cs.transform(xs, ys)` |
| World → data | `cs.to_data(x, y)` |
| Data group | `cs.data_group` |
| Annotations | `cs.vline(...)` / `cs.hline(...)` / `cs.line(...)` / `cs.point(...)` |
| Live trails | `cs.add_plot(path, auto_x=True)` + `cs.update_plots()` |
| In-place updates | `cs.xlim = ...` / `cs.yscale = "log"` / `cs.size = ...` |
| Styling | `x_style` / `y_style` / `grid_style` / `plane_style` |

**Next:** [11 — Labels](../11_labels/).
